# Pipeline Debug Walkthrough
Step through one sample: **load → technicals → prompt → decision → outcome**.

Set `SYMBOL`, `DATE`, `SLOT` in the first cell then **Run All**.


In [ ]:
# ── Parameters ─────────────────────────────────────────────────
SYMBOL     = "CANBK"        # NSE ticker
DATE       = "2024-08-19"   # decision date  YYYY-MM-DD
SLOT       = "open"         # "open" | "mid" | "close"
LOOKBACK   = 90             # days of candle history
CHART_BARS = 40             # bars shown in annotated charts


In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath("."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
import matplotlib.transforms as mtransforms

# ── TradingView-inspired dark palette ───────────────────────────
C = dict(
    bg="#131722", grid="#363a45", text="#d1d4dc",
    up="#26a69a",  dn="#ef5350",
    e50="#f6c90e", e100="#ff9800", e200="#f44336",
    sup="#26a69a", res="#ef5350",
    vwap="#b39ddb", entry="#42a5f5",
    sl="#ef5350",  tp="#26a69a",
    sh="#f6c90e",  sw="#7e57c2",
)
plt.rcParams.update({
    "figure.facecolor": C["bg"],  "axes.facecolor":  C["bg"],
    "axes.edgecolor":   C["grid"],"text.color":       C["text"],
    "xtick.color":      C["text"],"ytick.color":      C["text"],
    "grid.color":       C["grid"],"grid.linewidth":   0.5,
    "font.size": 9, "figure.dpi": 130,
})

# ── Chart helpers ───────────────────────────────────────────────
def _candles(ax, df):
    for i, (_, r) in enumerate(df.iterrows()):
        col = C["up"] if r.close >= r.open else C["dn"]
        lo, hi = min(r.open, r.close), max(r.open, r.close)
        ax.add_patch(Rectangle((i - 0.4, lo), 0.8, max(hi - lo, 0.001),
                                color=col, zorder=3))
        ax.plot([i, i], [r.low, r.high], color=col, lw=0.7, zorder=2)
    ax.set_xlim(-0.8, len(df) + 0.2)
    ax.autoscale_view()
    n = len(df); step = max(1, n // 8); ticks = list(range(0, n, step))
    ax.set_xticks(ticks)
    ax.set_xticklabels([df.index[i].strftime("%b %d") for i in ticks],
                        rotation=30, ha="right", fontsize=8)
    ax.yaxis.tick_right(); ax.yaxis.set_label_position("right")
    ax.grid(alpha=0.35)

def _hline(ax, y, color, label="", lw=1.0, ls="--"):
    ax.axhline(y, color=color, lw=lw, ls=ls, alpha=0.85, zorder=4)
    if label:
        trans = mtransforms.blended_transform_factory(ax.transAxes, ax.transData)
        ax.text(0.004, y, f"  {label}", transform=trans,
                color=color, fontsize=7.5, va="bottom", zorder=5)

def _xi(df, date_str):
    # numeric x-index of the bar closest to date_str
    target = pd.Timestamp(date_str)
    diffs  = [(abs((ts - target).days), i) for i, ts in enumerate(df.index)]
    return min(diffs)[1]

def _sw_marks(ax, points, chart_df, marker, color, dy):
    # plot swing high/low triangles inside the chart window
    chart_start = chart_df.index[0]
    for s in (points or []):
        if pd.Timestamp(s["date"]) >= chart_start:
            xi = _xi(chart_df, s["date"])
            ax.plot(xi, s["price"], marker, color=color, ms=7, zorder=6)
            ax.text(xi, s["price"] + dy, f"{s['price']:.1f}",
                    color=color, fontsize=6.5, ha="center", va="bottom", zorder=6)

from technicals import TechnicalsAnalyzer
from prompts import build_prompt, SYSTEM_PROMPT
ta = TechnicalsAnalyzer()
print("Imports OK — ta ready")


In [ ]:
# ══════════════════════════════════════════════════════════════
print(f"STEP 1 · Load {LOOKBACK}-day candle history")
print(f"  {SYMBOL}  {DATE}  @{SLOT}")
print("─" * 56)

df = ta.load_candles(SYMBOL, DATE, LOOKBACK)
print(f"  {len(df)} bars:  {df.index[0].date()} → {df.index[-1].date()}")
print(f"  Last close : {df['close'].iloc[-1]:.2f}")
print(f"  Period high: {df['high'].max():.2f}    low: {df['low'].min():.2f}")
print(f"  Avg vol    : {df['volume'].mean()/1e6:.1f}M/day")

# ── 90-day OHLCV + volume chart ─────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7),
    gridspec_kw={"height_ratios": [3, 1]}, sharex=False)
fig.suptitle(f"Step 1 · {SYMBOL} · {LOOKBACK}-day OHLCV  [{DATE} @{SLOT}]",
             color=C["text"], fontsize=11)

_candles(ax1, df)

n = len(df); step = max(1, n // 8); ticks = list(range(0, n, step))
vcols = [C["up"] if df["close"].iloc[i] >= df["open"].iloc[i] else C["dn"]
         for i in range(n)]
ax2.bar(range(n), df["volume"] / 1e6, color=vcols, alpha=0.7, width=0.8)
ax2.set_ylabel("Vol (M)", color=C["text"])
ax2.yaxis.tick_right(); ax2.yaxis.set_label_position("right"); ax2.grid(alpha=0.3)
ax2.set_xticks(ticks)
ax2.set_xticklabels([df.index[i].strftime("%b %d") for i in ticks],
                     rotation=30, ha="right", fontsize=8)

di = _xi(df, DATE)
for _ax in (ax1, ax2):
    _ax.axvline(di, color="#ffffff", lw=0.8, ls=":", alpha=0.5)
ax1.text(di + 0.4, ax1.get_ylim()[1], f"  {DATE}", color=C["text"],
         fontsize=7.5, va="top")
plt.tight_layout(); plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════
print("STEP 2 · Compute technical indicators")
print("─" * 56)

tech = ta.compute(df, decision_slot=SLOT)
vc   = ta.vix_context(DATE)
zc   = ta.compute_zones(SYMBOL, DATE)
if vc: tech["vix_ctx"] = vc
if zc: tech["zones"]   = zc

close = tech["close"]
print(f"  Close  : {close:.2f}    VWAP: {tech['vwap']}")
print(f"  EMA50  : {tech['ema_50']}   EMA100: {tech['ema_100']}   EMA200: {tech['ema_200']}")
print(f"  Stack  : {tech['ema_stack']}")
print(f"  Trend  : {tech['trend']}    Struct: {tech.get('structure') or '-'}")
ns = tech.get("nearest_support"); nr = tech.get("nearest_resistance")
print(f"  Nearest S: {ns['mid'] if ns else '—'}   Nearest R: {nr['mid'] if nr else '—'}")
szones = [z['mid'] for z in (tech.get('support_zones') or [])]
rzones = [z['mid'] for z in (tech.get('resistance_zones') or [])]
print(f"  S-zones : {szones}")
print(f"  R-zones : {rzones}")
if vc:
    print(f"  VIX     : {vc['vix']}  ({vc['regime']})  pctile_1y={vc['pctile_1y']}%")
rc = tech.get("recent_candles") or []
print(f"  Recent candles : {[r['date'] for r in rc]}")
ok = (rc[-1]['date'] < DATE) if rc and SLOT in ("open","mid") else True
print(f"  Look-ahead OK  : {ok}  (last_candle={rc[-1]['date'] if rc else 'n/a'}, decision={DATE})")


In [ ]:
# ── Build chart dataframe (what the LLM sees) ──────────────
if SLOT in ("open", "mid"):
    view_df = df[df.index < pd.Timestamp(DATE)]
else:
    view_df = df.copy()
chart_df = view_df.tail(CHART_BARS)

# Rolling EMAs on full df for smooth lines, sliced to chart window
_e50  = df["close"].ewm(span=50,  adjust=False).mean().loc[chart_df.index]
_e100 = df["close"].ewm(span=100, adjust=False).mean().loc[chart_df.index]
_e200 = df["close"].ewm(span=200, adjust=False).mean().loc[chart_df.index]

fig, ax = plt.subplots(figsize=(14, 7))
fig.suptitle(f"Step 2 · {SYMBOL} · Technical setup · {DATE} @{SLOT}",
             color=C["text"], fontsize=11)
_candles(ax, chart_df)
xs = list(range(len(chart_df)))

# EMAs
ax.plot(xs, _e50.values,  color=C["e50"],  lw=1.3, label=f"EMA50  {tech['ema_50']:.1f}")
ax.plot(xs, _e100.values, color=C["e100"], lw=1.3, label=f"EMA100 {tech['ema_100']:.1f}")
ax.plot(xs, _e200.values, color=C["e200"], lw=1.3, label=f"EMA200 {tech['ema_200']:.1f}")

# VWAP
ax.axhline(tech["vwap"], color=C["vwap"], lw=1.1, ls="-.", alpha=0.85, zorder=4,
           label=f"VWAP {tech['vwap']:.2f}")

# Support / resistance zones (shaded bands)
for z in (tech.get("support_zones") or []):
    lo = z["low"]; hi = max(z["high"], z["low"] + 0.1)
    ax.axhspan(lo - 0.02, hi + 0.02, color=C["sup"], alpha=0.09, zorder=1)
for z in (tech.get("resistance_zones") or []):
    lo = z["low"]; hi = max(z["high"], z["low"] + 0.1)
    ax.axhspan(lo - 0.02, hi + 0.02, color=C["res"], alpha=0.09, zorder=1)

# Nearest S/R lines
if ns: _hline(ax, ns["mid"], C["sup"], f"S {ns['mid']:.2f}", lw=1.4)
if nr: _hline(ax, nr["mid"], C["res"], f"R {nr['mid']:.2f}", lw=1.4)

# PDH / PDL
lv = tech.get("levels") or {}
if lv.get("pdh"): _hline(ax, lv["pdh"], "#90a4ae", f"PDH {lv['pdh']}", lw=0.8, ls=":")
if lv.get("pdl"): _hline(ax, lv["pdl"], "#90a4ae", f"PDL {lv['pdl']}", lw=0.8, ls=":")

# Swing highs (yellow ▲) / lows (purple ▼)
sh_pts = tech.get("swing_highs_30d") or tech.get("swing_highs")
sl_pts = tech.get("swing_lows_30d")  or tech.get("swing_lows")
_sw_marks(ax, sh_pts, chart_df, "^", C["sh"],  dy= 0.4)
_sw_marks(ax, sl_pts, chart_df, "v", C["sw"],  dy=-0.8)

# Structure labels (HH/LH/HL/LL) near the right edge
struct = tech.get("structure") or []
if struct:
    ax.text(0.98, 0.02, "  ".join(struct), transform=ax.transAxes,
            color=C["text"], fontsize=9, ha="right", va="bottom", alpha=0.7)

# VIX corner tag
if vc:
    ax.text(0.01, 0.98, f"VIX {vc['vix']} · {vc['regime']} · {vc.get('dir_5d','')}",
            transform=ax.transAxes, color=C["vwap"], fontsize=8.5, va="top", alpha=0.8)

ax.legend(loc="lower left", fontsize=8, framealpha=0.3,
          facecolor=C["bg"], edgecolor=C["grid"])
plt.tight_layout(); plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════
print("STEP 3 · Prompt fed to the LLM")
print("─" * 56)
prompt_text = build_prompt(SYMBOL, DATE, tech, vc or {}, zc or {}, SLOT)
print(prompt_text)


In [ ]:
# ══════════════════════════════════════════════════════════════
print("STEP 4 · LLM / rule-based decision")
print("─" * 56)
try:
    from pipeline import decide
    dec, _ = decide(SYMBOL, DATE, tech, vc or {}, zc or {}, decision_slot=SLOT)
except Exception as _err:
    print(f"[LLM unavailable: {_err}]  — storing HOLD placeholder")
    dec = {"action": "HOLD", "planned_entry": tech["close"],
           "stop_loss": 0, "take_profit": 0, "risk_reward": 0,
           "confidence": 0, "reasoning": "LLM not reachable", "features_used": []}

action = dec.get("action", "HOLD")
entry  = float(dec.get("planned_entry") or tech["close"])
sl     = float(dec.get("stop_loss")     or 0)
tp     = float(dec.get("take_profit")   or 0)
conf   = float(dec.get("confidence")    or 0)
rr     = float(dec.get("risk_reward")   or 0)

print(json.dumps(dec, indent=2))
print()
print(f"  ACTION : {action}")
print(f"  Entry  : {entry:.2f}")
if sl: print(f"  SL     : {sl:.2f}  ({(sl - entry) / entry * 100:+.2f}%)")
if tp: print(f"  TP     : {tp:.2f}  ({(tp - entry) / entry * 100:+.2f}%)")
print(f"  Conf   : {conf:.2f}    R:R : {rr:.2f}")


In [ ]:
# ── Entry / SL / TP chart (last 20 LLM-visible bars) ────────
dec_df = chart_df.tail(20)

fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle(
    f"Step 5 · {SYMBOL} · {action}  "
    + (f"entry={entry:.2f}  SL={sl:.2f}  TP={tp:.2f}  R:R={rr:.2f}"
       if action != "HOLD" else "HOLD — no trade"),
    color=C["text"], fontsize=11,
)
_candles(ax, dec_df)

if action != "HOLD":
    _hline(ax, entry, C["entry"], f"ENTRY {entry:.2f}", lw=1.6, ls="-")
    if sl: _hline(ax, sl, C["sl"], f"SL  {sl:.2f}", lw=1.2)
    if tp: _hline(ax, tp, C["tp"], f"TP  {tp:.2f}", lw=1.2)

    # Risk/reward shading
    if sl and tp:
        ax.axhspan(min(entry,sl), max(entry,sl), color=C["sl"], alpha=0.07, zorder=1)
        ax.axhspan(min(entry,tp), max(entry,tp), color=C["tp"], alpha=0.07, zorder=1)

# Big action label
acol = C["tp"] if action == "BUY" else C["sl"] if action == "SELL" else C["text"]
ax.text(0.99, 0.97, f"▶ {action}", transform=ax.transAxes, color=acol,
        fontsize=20, fontweight="bold", ha="right", va="top", alpha=0.75)

# Reasoning snippet
rsn = (dec.get("reasoning") or "").strip()
if rsn and rsn != "LLM not reachable":
    snippet = rsn[:160] + ("…" if len(rsn) > 160 else "")
    ax.text(0.01, 0.02, f'"{snippet}"', transform=ax.transAxes,
            color=C["text"], fontsize=7.5, va="bottom", alpha=0.7,
            bbox=dict(facecolor=C["bg"], edgecolor=C["grid"], alpha=0.85, pad=3))

ax.legend(loc="upper left", fontsize=8, framealpha=0.3,
          facecolor=C["bg"], edgecolor=C["grid"])
plt.tight_layout(); plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════
print("STEP 6 · Simulate outcome")
print("─" * 56)
try:
    from pipeline import simulate_outcome, MAX_HOLD_DAYS, compute_pnl, trade_costs
    fwd = ta.load_forward(SYMBOL, DATE, MAX_HOLD_DAYS)
    exit_reason, exit_p, exit_date, _ = simulate_outcome(
        action, entry, sl, tp, fwd,
        max_hold=MAX_HOLD_DAYS, planned_entry=entry,
    )
    if exit_reason not in ("ENTRY_NOT_FILLED", "HOLD") and action != "HOLD" and sl and tp:
        gross = compute_pnl(action, entry, exit_p, 1)
        pnl   = round(gross - trade_costs(entry, exit_p, 1, action), 2)
    else:
        pnl = 0.0

    print(f"  Exit reason : {exit_reason}")
    print(f"  Exit price  : {exit_p:.2f}  (on {exit_date})")
    print(f"  P&L (qty=1) : {pnl:+.2f}")

    # ── Resample forward bars to daily for charting ──────────
    if len(fwd) > MAX_HOLD_DAYS * 2:          # minute bars
        fwd_d = (fwd.resample("1D")
                   .agg({"open":"first","high":"max","low":"min",
                         "close":"last","volume":"sum"})
                   .dropna(subset=["close"]))
    else:
        fwd_d = fwd.copy()

    # Combine: last 10 historical bars + forward bars
    hist_part = view_df.tail(10).copy()
    fwd_d.index = fwd_d.index.normalize()
    hist_part.index = hist_part.index.normalize()
    fwd_only = fwd_d[~fwd_d.index.isin(hist_part.index)]
    combo = pd.concat([hist_part, fwd_only])
    n_hist = len(hist_part)

    fig, ax = plt.subplots(figsize=(14, 7))
    mark = "PROFIT" if pnl > 0 else "LOSS" if pnl < 0 else "FLAT"
    fig.suptitle(
        f"Step 6 · {SYMBOL} · {exit_reason} @ {exit_p:.2f}  [{mark}  {pnl:+.2f}]",
        color=C["text"], fontsize=11,
    )
    _candles(ax, combo)

    # Divider between history and forward
    ax.axvline(n_hist - 0.5, color="#ffffff", lw=0.9, ls="--", alpha=0.4, zorder=5)
    ax.text(n_hist - 0.3, ax.get_ylim()[1], "  → entry day", color=C["text"],
            fontsize=7.5, va="top")

    # Entry / SL / TP lines across full chart
    if action != "HOLD":
        ax.axhline(entry, color=C["entry"], lw=1.5, ls="-",  alpha=0.9, zorder=4,
                   label=f"Entry {entry:.2f}")
        if sl: ax.axhline(sl, color=C["sl"], lw=1.2, ls="--", alpha=0.85, zorder=4,
                          label=f"SL {sl:.2f}")
        if tp: ax.axhline(tp, color=C["tp"], lw=1.2, ls="--", alpha=0.85, zorder=4,
                          label=f"TP {tp:.2f}")

    # Exit star marker
    if exit_date:
        try:
            xi = _xi(combo, exit_date)
            ecol = C["tp"] if exit_reason == "TP_HIT" else C["sl"] if exit_reason == "SL_HIT" else C["entry"]
            ax.plot(xi, exit_p, "*", color=ecol, ms=16, zorder=7, label=f"{exit_reason}")
            ax.annotate(f"  {exit_reason}\n  {exit_p:.2f}",
                        xy=(xi, exit_p), xytext=(xi + 0.5, exit_p),
                        color=ecol, fontsize=8, va="center", zorder=7,
                        arrowprops=dict(arrowstyle="-", color=ecol, lw=0.7))
        except Exception:
            pass

    # P&L badge
    pcol = C["tp"] if pnl >= 0 else C["sl"]
    ax.text(0.99, 0.97, f"P&L  {pnl:+.2f}",
            transform=ax.transAxes, color=pcol,
            fontsize=16, fontweight="bold", ha="right", va="top", alpha=0.85)

    ax.legend(loc="upper left", fontsize=8, framealpha=0.3,
              facecolor=C["bg"], edgecolor=C["grid"])
    plt.tight_layout(); plt.show()

except Exception as _err:
    import traceback
    print(f"[outcome simulation failed: {_err}]")
    traceback.print_exc()
